# Flow Matching

The flow matching transformer model is a flexible and stochastic
surrogate.

## When this might be a good fit

The flow matching surrogate is a good option when stochasticity is
needed. Architecturally, it is implemented as a modified transformer and
also supports static and sequence conditioning.

## How flow matching works

### Architecture

The flow matching surrogate is parametrised with a *non-causal*
transformer that processes the entire sequence in parallel.

### Flow Matching

Flow Matching and Diffusion models are the current state-of-the-art
learning algorithms for continuous valued data (e.g. images,
protein-structure prediction). In brief, these models learn time-varying
velocity fields which can be used to transport a sample of pure noise
(e.g. from a Gaussian) to the data distribution. In order to generate
samples the time-varying velocity field $v_{\theta}$ must be integrated
over time $t$ to transport a noise sample $x_0$ to the data
distribution: $$
x_{\mathrm{pred}} = x_0+\int_{t=0}^{t=1}v_{\theta}(x_t|c)dt\text{, where } c \text{ is conditioning (i.e. surrogate inputs)}
$$

In practice this is achieved via numerical ODE solvers like Euler or
Heun, with the number of solver steps introducing a trade-off between
speed and quality.

> **Technical note**
>
> Diffusion models have equivalent ODE and SDE forms and can be sampled
> via SDE-solvers like Euler-Maruyama or ODE-solvers like Heun. See Song
> et al. (2021).

Flow matching models iteratively transform samples from noise to the
data distribution. Animation produced by Floor Eijkelboom.

By fixing the conditioning $c$ and varying the initial noise, you can
sample multiple realisations rather than the average realisation that an
MLP or Autoregressive transformer learn.

> **Tip**
>
> The mathematical details of these models and the derivation of their
> training objectives are out of scope but if you’re interested you may
> find these resources helpful:
>
> -   [Accessible introduction to Diffusion
>     (YouTube)](https://www.youtube.com/watch?v=iv-5mZ_9CPY)
> -   [Introduction to Normalising Flows and Flow Matching
>     (Blog)](https://mlg.eng.cam.ac.uk/blog/2024/01/20/flow-matching.html)
> -   [Explanation of score-based diffusion by its inventor
>     (Blog)](https://yang-song.net/blog/2021/score/#connection-to-diffusion-models-and-others)
> -   [Unifying flow matching and diffusion
>     (Monograph)](https://arxiv.org/pdf/2510.21890)

## Configuring a Flow Matching model

Metis’s implementation is designed to support different kinds of
*Stochastic Interpolant* algorithms (the general name for diffusion and
flow matching), although currently only Flow Matching has been set up
and tested. The top-level config controls details like the type of
algorithm and solver steps for evaluation and contains an inner
architectural config that is very similar to the transformer:

-   `interpolant`: Currently only ‘flow-matching’ supported.
-   `num_steps`: Number of ODE solver steps for evaluations, does not
    impact training.
-   `solver`: ODE solver, currently only ‘heun’ (second-order) and
    ‘euler’ (first-order) are supported.
-   `multi_mode`: Whether to generate multiple samples from the median
    of the parameter group (‘median’ option) or to generate multiple
    samples for each entry in the parameter group
    (‘entire_parameter_group’). This second option only makes sense for
    models like HRP and Magenta where realisations within the same
    parameter group can have different static inputs.
-   `sit`:
    -   `sequence_len`: Number of timesteps in the data sequence.
    -   `data_dim`: Number of variables per datapoint.
    -   `cond_dim`: Number of static input variables.
    -   `n_blocks`: Number of transfomer blocks.
    -   `n_head`: Number of query heads in attention layers.
    -   `n_kv_head`: Number of key-value heads in attention layers.
    -   `qkv_bias`: Whether the projection matrices in the
        self-attention module use biases.
    -   `emdb_dim`: Hidden dimension of the network
    -   `cond_dropout_prob`: Currently unused - but could be used to
        enable [classifier-free
        guidance](https://arxiv.org/abs/2207.12598)
    -   `in_and_out_biases`: Whether the input and out projections use
        learnable biases.
    -   `cross_attn`: The cross attention config for sequence
        conditioning, can be `None`.

The `sequence_len`, `data_dim` and `cond_dim` are all automatically
configured based on the dataset config.

| Config                  | `n_blocks` | `n_head` | `n_kv_head` | `embd_dim` |     |
|-------------------------|-----------:|---------:|------------:|-----------:|-----|
| `flow_matching_small`   |          6 |        4 |           4 |         64 |     |
| `flow_matching_default` |         12 |        8 |           8 |         64 |     |
| `flow_matching_large`   |         16 |       16 |          16 |        128 |     |

Song, Yang, Jascha Sohl-Dickstein, Diederik P Kingma, Abhishek Kumar,
Stefano Ermon, and Ben Poole. 2021. “Score-Based Generative Modeling
Through Stochastic Differential Equations.” In *International Conference
on Learning Representations*. <https://arxiv.org/abs/2011.13456>.